# Battle Tailscale Worker（单 cell）

Tailscale 内网直连，替代 cloudflared。**改 CFG 后点 Run 即就绪。**

| mode | rl_mode | 说明 | 本地要做什么 |
|---|---|---|---|
| `rl` | `push` | Colab 起 worker，本地推 PPO job | rl-config 节点 url → `http://<TS_IP>:8790` |
| `rl` | `pull` | Colab 轮询本地 hub | hub 跑在本地 Tailscale IP |
| `bc` | — | Colab 本地 BC 蒸馏 | 上传语料 zip |

RL 运行时逻辑在 code.zip（`remote/notebook_runtime.py`）；BC 逻辑在 `remote/colab_bc.py`。

**停止**：■ 中断本 cell。


In [ ]:
# @title Battle Tailscale Worker — 改参数后点 Run
import os, sys, threading, time, subprocess, json, shutil, zipfile, io, base64, secrets
from pathlib import Path
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

CFG = {
    "mode": "rl",                # rl | bc
    "rl_mode": "push",           # push | pull
    "ts_authkey": "",            # 留空=已登录; 填 tskey-auth-xxx
    "push_port": 8790,
    "push_token": "YOUR_TOKEN_HERE",
    "hub_url": "",               # pull: http://<本地TS_IP>:8787
    "hub_token": "",
    "device": "auto",
    "max_session_hours": 9,
    # BC
    "bc_run_tag": "p3bc",
    "bc_course": "p3-bc",
    "bc_corpus_zip": "p3-godai.zip",
    "bc_epochs": 60,
    "bc_seed": 1234,
    "repo_url": "https://github.com/HuangJian/battle.git",
    "branch": "goal-nn",
}

def _log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] [battle] {msg}", flush=True)

_log(f"mode={CFG['mode']}" + (f"/{CFG['rl_mode']}" if CFG['mode']=='rl' else ""))

# ── Keepalive ──────────────────────────────────────────────
_keepalive_stop = threading.Event()
def _keepalive_loop():
    if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ:
        while not _keepalive_stop.is_set():
            try:
                from IPython.display import Javascript, display
                display(Javascript(
                    "function f(){document.querySelector('colab-connect-button')?.click();}"
                    "setTimeout(f,1000);"))
            except Exception: pass
            _keepalive_stop.wait(60)
    else:
        n = 0
        while not _keepalive_stop.is_set():
            print(f"[{time.strftime('%H:%M:%S')}] [keepalive] alive ({n*3} min)", flush=True)
            n += 1; _keepalive_stop.wait(180)
threading.Thread(target=_keepalive_loop, daemon=True, name="keepalive").start()
_log("Keepalive 已启动")

# ── Tailscale ──────────────────────────────────────────────
if not shutil.which("tailscale"):
    _log("安装 Tailscale …")
    subprocess.run("curl -fsSL https://tailscale.com/install.sh | sh",
                   shell=True, check=True, timeout=180)
_r = subprocess.run(["tailscale", "ip", "-4"], capture_output=True, text=True)
if _r.returncode == 0 and _r.stdout.strip():
    TS_IP = _r.stdout.strip()
else:
    _ak = str(CFG.get("ts_authkey") or "").strip()
    if _ak:
        subprocess.run(["tailscale", "up", f"--authkey={_ak}"], check=True, timeout=60)
    else:
        subprocess.run(["tailscale", "up"], timeout=120)
    for _ in range(30):
        _r = subprocess.run(["tailscale", "ip", "-4"], capture_output=True, text=True)
        if _r.returncode == 0 and _r.stdout.strip(): TS_IP = _r.stdout.strip(); break
        time.sleep(1)
    else:
        raise RuntimeError("Tailscale 未能获取 IP")
_log(f"Tailscale IP = {TS_IP}")

# ══════════════════════════════════════════════════════════
# RL
# ══════════════════════════════════════════════════════════
if CFG["mode"] == "rl":
    _rl_mode = str(CFG["rl_mode"] or "push").lower()
    _port = int(CFG["push_port"])
    _tok = str(CFG["push_token"])
    _code_dir = "/tmp/worker-code"
    _work = Path("/tmp/remote-worker-serve"); _work.mkdir(parents=True, exist_ok=True)
    _hub = str(CFG.get("hub_url") or "").strip()

    # Pull / push-with-hub: GET /code
    if _rl_mode == "pull" or (_rl_mode == "push" and _hub):
        if not _hub: raise SystemExit("[FATAL] pull 需要 hub_url")
        _deadline = time.time() + 3600
        while True:
            try:
                from urllib.request import Request, urlopen
                _req = Request(_hub.rstrip("/") + "/code",
                    headers={"Authorization": "Bearer " + str(CFG.get("hub_token") or "")})
                with urlopen(_req, timeout=120) as _resp: _raw = _resp.read()
                _log(f"code.zip 就绪: {len(_raw)} bytes"); break
            except Exception as _e:
                if time.time() > _deadline: raise SystemExit("[FATAL] 等 code.zip 超 1h") from None
                _log(f"hub 异常（{type(_e).__name__}）—— 30s 重试"); time.sleep(30)
        Path(_code_dir).mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(io.BytesIO(_raw)) as _z: _z.extractall(_code_dir)
        sys.path.insert(0, _code_dir)
        from remote.notebook_runtime import run_notebook
        raise SystemExit(run_notebook({
            "mode": _rl_mode, "hub_url": _hub, "hub_token": str(CFG.get("hub_token") or ""),
            "push_port": _port, "push_token": _tok, "device": CFG["device"],
            "use_multi_gpu": True, "max_session_hours": CFG["max_session_hours"],
            "poll_interval_sec": 1, "idle_floor_sec": 3600, "max_worker_restarts": 5,
            "keepalive_stop": _keepalive_stop, "log": _log, "code_dir": _code_dir,
        }))

    # Push-first: 内联 bootstrap（首包前无 code.zip）
    if _tok in ("", "YOUR_TOKEN_HERE"): raise SystemExit("[FATAL] push 需要 push_token")
    _cdir = Path(_code_dir); _upg = {"body": None}

    class _H(BaseHTTPRequestHandler):
        def log_message(self, *a): pass
        def _j(self, o, s=200):
            b = json.dumps(o).encode(); self.send_response(s)
            self.send_header("Content-Type", "application/json")
            self.send_header("Content-Length", str(len(b))); self.end_headers(); self.wfile.write(b)
        def _ok(self):
            return secrets.compare_digest(self.headers.get("Authorization", ""), f"Bearer {_tok}")
        def do_GET(self):
            if not self._ok(): return self._j({"error": "unauthorized"}, 401)
            p = self.path.split("?", 1)[0]
            if p == "/ping": return self._j({"ok": True, "busy": False, "queued": 0, "done": 0, "bootstrap": True})
            if p == "/code-sha": return self._j({"cached": False})
            return self._j({"error": "nf"}, 404)
        def do_POST(self):
            if not self._ok(): return self._j({"error": "unauthorized"}, 401)
            if self.path.split("?", 1)[0] != "/job": return self._j({"error": "nf"}, 404)
            raw = self.rfile.read(int(self.headers.get("Content-Length", "0")))
            try: body = json.loads(raw.decode())
            except ValueError: return self._j({"error": "bad json"}, 400)
            if not body.get("code_b64"): return self._j({"error": "code-missing"}, 428)
            try:
                _cdir.mkdir(parents=True, exist_ok=True)
                zipfile.ZipFile(io.BytesIO(base64.b64decode(body["code_b64"]))).extractall(_cdir)
            except Exception as e: return self._j({"error": str(e)}, 400)
            _upg["body"] = raw; _log(f"code.zip 已解包 -> {_cdir}")
            return self._j({"status": "accepted", "upgrading": True}, 202)

    _srv = ThreadingHTTPServer(("0.0.0.0", _port), _H)
    threading.Thread(target=_srv.serve_forever, daemon=True).start()
    _log(f"push bootstrap :{_port}")
    _log(f"★ rl-config 节点 url -> http://{TS_IP}:{_port}  (authKey=push_token)")
    _deadline = time.time() + int(CFG["max_session_hours"]) * 3600
    try:
        while _upg["body"] is None:
            if time.time() > _deadline: raise SystemExit(0)
            time.sleep(2)
    except KeyboardInterrupt:
        _log("中断"); raise SystemExit(0)
    finally:
        _srv.shutdown(); _srv.server_close(); time.sleep(0.5)

    # 升级到完整 worker_server
    sys.path.insert(0, _code_dir)
    from remote.push_bootstrap import requeue_job, spawn_full_worker_server, wait_ping
    from remote.notebook_runtime import resolve_device, run_notebook
    _rl_cfg = {
        "mode": "push", "push_port": _port, "push_token": _tok,
        "device": CFG["device"], "use_multi_gpu": True,
        "max_session_hours": CFG["max_session_hours"],
        "poll_interval_sec": 1, "idle_floor_sec": 3600, "max_worker_restarts": 5,
        "keepalive_stop": _keepalive_stop, "log": _log, "code_dir": _code_dir,
    }
    _rl_cfg["device_resolved"] = resolve_device(_rl_cfg, _log)
    _real = spawn_full_worker_server(
        _port, _tok, _work, str(_rl_cfg["device_resolved"]), _cdir, _work / "serve.log")
    if not wait_ping(_port, _tok, 30):
        _log("worker_server 30s 未就绪——serve.log 尾部:")
        try:
            for _ln in (_work / "serve.log").read_text(encoding="utf-8", errors="replace").splitlines()[-40:]:
                _log(f"  | {_ln}")
        except OSError: pass
        _real.kill(); raise SystemExit(-1)
    _log("worker_server 就绪——重放首个 job 后守候")
    requeue_job(_port, _tok, _upg["body"] or b"{}")
    _rl_cfg["already_serving"] = {
        "serve_pid": _real.pid, "cf_pid": None,  # Tailscale: 无 cloudflared
        "cf_url": f"http://{TS_IP}:{_port}",
    }
    raise SystemExit(run_notebook(_rl_cfg))

# ══════════════════════════════════════════════════════════
# BC — 薄壳: Drive + clone + 调 remote/colab_bc.py
# ══════════════════════════════════════════════════════════
elif CFG["mode"] == "bc":
    _repo = Path("/content/battle2")
    _branch = CFG["branch"]
    if shutil.which("git") and (_repo / ".git").exists():
        _log("git pull …")
        subprocess.run(["git", "-C", str(_repo), "fetch", "--depth", "1", "origin", _branch], check=True)
        subprocess.run(["git", "-C", str(_repo), "checkout", "-B", _branch, "FETCH_HEAD"], check=True)
    else:
        _log("clone …")
        _repo.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch", _branch,
                        CFG["repo_url"], str(_repo)], check=True)
    sys.path.insert(0, str(_repo / "nn-training"))
    from remote.colab_bc import main as bc_main
    raise SystemExit(bc_main([
        "--run-tag", CFG["bc_run_tag"], "--course", CFG["bc_course"],
        "--corpus-zip", CFG["bc_corpus_zip"], "--epochs", str(CFG["bc_epochs"]),
        "--seed", str(CFG["bc_seed"]), "--repo", str(_repo),
    ]))

else:
    raise SystemExit(f"[FATAL] 未知 mode={CFG['mode']!r}")
